# День 1 · стоимость и контекстное окно

Две вещи меряем прайсом одного и того же `GET /api/v1/models`: во сколько обходится сценарий на
твоей модели против дешёвой альтернативы, и сколько токенов на самом деле съедает контекст —
system-промпт, история и документы делят один бюджет с ответом модели.

In [ ]:
import httpx
import labkit

from client import BASE_URL, MODEL, make_client

# --- НАСТРОЙКИ ---
IN_TOKENS, OUT_TOKENS = 3500, 400           # типичный RAG-запрос: контекст + вопрос → короткий ответ
REQ_PER_MONTH = 1000 * 30                   # нагрузка сценария: тысяча запросов в день
CHEAP_MODEL = labkit.env("LLM_MODEL_CHEAP", "deepseek/deepseek-v4-flash-0731")  # модель для сравнения
MIN_CONTEXT = 100_000                       # фильтр для списка дешёвых моделей

## Каталог моделей: цены и контекст в одном месте

In [ ]:
def load_models() -> list[dict]:
    """Каталог моделей OpenRouter: цены, размер контекста, поддерживаемые параметры."""
    r = httpx.get(f"{BASE_URL}/models", timeout=30)  # прокси из HTTPS_PROXY подхватится сам
    r.raise_for_status()                    # исключение при HTTP-ошибке
    return r.json()["data"]


def price(models: list[dict], model_id: str) -> tuple[float, float, int | None]:
    """Цена за один токен входа и выхода (USD) и размер контекста."""
    for m in models:
        if m["id"] == model_id:
            p = m["pricing"]
            return float(p["prompt"]), float(p["completion"]), m.get("context_length")
    raise SystemExit(f"модель {model_id} не найдена в /models")


def monthly_cost(p_in: float, p_out: float) -> float:
    return (IN_TOKENS * p_in + OUT_TOKENS * p_out) * REQ_PER_MONTH

## Сравнение стоимости: основная модель против дешёвой

In [ ]:
models = load_models()
cheap_candidates = sorted(                                                        # платные модели с большим контекстом
    (m for m in models if float(m["pricing"]["prompt"]) > 0 and (m.get("context_length") or 0) >= MIN_CONTEXT),
    key=lambda m: float(m["pricing"]["prompt"]),                                  # сортировка по цене входа
)[:10]
print(f"10 самых дешёвых моделей с контекстом ≥ {MIN_CONTEXT // 1000}k (цена за 1M входных токенов, USD):")
for m in cheap_candidates:
    print(f"  {float(m['pricing']['prompt']) * 1e6:8.3f}  {m['id']}")

for label, mid in (("основная", MODEL), ("дешёвая", CHEAP_MODEL)):
    p_in, p_out, ctx = price(models, mid)
    print(f"\n{label}: {mid} (контекст {ctx})")
    print(f"  вход ${p_in * 1e6:.2f}/1M, выход ${p_out * 1e6:.2f}/1M, отношение выход/вход: {p_out / p_in if p_in else None}")
    print(f"  один RAG-запрос: ${IN_TOKENS * p_in + OUT_TOKENS * p_out:.5f}")
    print(f"  {REQ_PER_MONTH} запросов в месяц: ${monthly_cost(p_in, p_out):.2f}")

## Контекстное окно: один бюджет на всё

Системный промпт, история диалога, найденные документы (RAG, день 2) и сам ответ — всё это делит
один лимит токенов, контекстное окно модели, `context_length` из каталога выше. Ниже — не теория,
а измерение: берём один и тот же абзац, повторяем его 1, 20 и 200 раз, отправляем с `max_tokens=1`
(ответ не нужен — нужен только точный счётчик входа) и смотрим, сколько токенов реально съел каждый
вход и какую долю окна он занял.

In [ ]:
client = make_client()
_, _, context_length = price(models, MODEL)
paragraph = "Ollama слушает порт 11434 и отдаёт OpenAI-совместимый API для локальных моделей. "
print(f"контекстное окно {MODEL}: {context_length} токенов\n")
for repeats in (1, 20, 200):
    text = paragraph * repeats
    r = client.chat.completions.create(model=MODEL, max_tokens=1, messages=[{"role": "user", "content": text}])
    used = r.usage.prompt_tokens
    pct = 100 * used / context_length if context_length else None
    print(f"текст ×{repeats:3d} ({len(text):6d} символов) → {used:6d} токенов входа ({pct:.2f}% окна)")

В настоящем запросе к этому тексту ещё добавятся системный промпт и история диалога — все они
считаются в ту же сумму. Превысишь окно — провайдер либо обрежет часть сообщения, либо вернёт
ошибку контекста; молча «раздвинуть» лимит нельзя ни для одной модели.